# Download the file instead

Notebook 1 pulled a file's *content* into the conversation. Sometimes you do not
want that. You just want the file saved, so you can open it yourself or keep it
for later.

Same CMS, same package, one different tool: `download_file` saves to disk and
tells you where it landed. The agent never reads the content.

## 1. Log in (same as before)

This works for **GIU** as well: put `GUC_SITE=giu` in your `.env` and use your
GIU username and password. Nothing else in the notebook changes.

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()  # reads ANTHROPIC_API_KEY from .env (and GUC login if you put it there)

# Type your GUC login once. getpass hides the password so it is never saved
# in the notebook. Nothing here is written to disk.
if not os.environ.get("GUC_USERNAME"):
    os.environ["GUC_USERNAME"] = input("GUC username: ")
if not os.environ.get("GUC_PASSWORD"):
    os.environ["GUC_PASSWORD"] = getpass.getpass("GUC password: ")

from guc_cms import GucCms

cms = GucCms()
courses = cms.list_courses()
print(len(courses), "courses. First few:")
for c in courses[:5]:
    print("  ", c.code, "-", c.title)

## 2. One tool: save to disk

`download_file` calls `cms.download`, which streams the file into a folder and
returns the path. Compare it to `read_course_file` from notebook 1: that returned
*text*, this returns a *path*.

In [ ]:
def _find_item(course, file_title):
    """Find one file in a course by a bit of its title. Returns a ContentItem or None."""
    c = cms.find_course(course)
    if not c:
        return None
    for item in cms.get_content(c).items:
        if file_title.lower() in item.title.lower():
            return item
    return None


from langchain.tools import tool


@tool
def get_course_files(course: str) -> list[dict]:
    """List the files in a course, so you can find a file's exact title."""
    c = cms.find_course(course)
    if not c:
        return [{"error": f"no course matches {course!r}"}]
    return [{"title": i.title, "kind": i.kind, "week": i.week}
            for i in cms.get_content(c).items]


@tool
def download_file(course: str, file_title: str, folder: str = "downloads") -> str:
    """Download a course file to disk and return the saved path.
    Does NOT read the content. `file_title` can be part of the title."""
    item = _find_item(course, file_title)
    if not item:
        return f"No file matching {file_title!r} in {course!r}."
    path = cms.download(item, folder)
    return f"Saved to {path}"

## 3. Give it to an agent

In [ ]:
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

model = ChatAnthropic(model="claude-haiku-4-5", temperature=0)

agent = create_agent(
    model=model,
    tools=[get_course_files, download_file],
    system_prompt="You help a GUC student download course files. Report the saved path.",
)

result = agent.invoke({"messages": [HumanMessage(
    "Download the PA 9 assignment from Software Engineering.")]})
print(result["messages"][-1].content)

import os
print("\nfiles now in downloads/:", os.listdir("downloads"))

## Context vs a file on disk

Two notebooks, two different moves:

|  | Notebook 1: read into context | Notebook 2: download |
| --- | --- | --- |
| the tool returns | the text | a file path |
| agent can discuss the content | yes | no |
| cost | tokens, grows with file size | almost none |
| good for | questions, summaries | keeping files, big or many |

Notebook 2's agent moved a file it never looked at. If we now want it to *work
with* those files (open them, list the folder, write a new one) it needs a proper
place to do that: a backend the agent is allowed to touch.

That is the next step.